# 05 — Text-only baseline without imbalance handling

**Input:** Member 1's processed CSV and explicit column/label mapping below. Remove duplicate or overlapping articles *before splitting* and agree on the real label mapping with the team. Do not include review verdicts, post-publication outcomes, or any other target-derived column as features.

The same fixed 60/20/20 stratified split and TF-IDF + logistic regression settings are used in notebook 06. The vectorizer is fitted on training rows only. Model selection belongs on validation; the held-out test is used for a single final comparison. No real data or execution outputs are checked into Git.


In [ ]:
from pathlib import Path
import sys

# Run Jupyter from the repository root or the notebooks directory.
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if not (ROOT / "src").exists():
    raise RuntimeError("Start Jupyter in the CrossHealth-Risk repository")
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.models.baseline import build_baseline, split_labeled_data
from src.evaluation.metrics import evaluate_binary_classifier
from src.evaluation.plots import plot_confusion, plot_roc_and_pr


## Set the actual processed file and label mapping


In [ ]:
# Fill these using Member 1's actual standardized output. No columns are assumed.
DATA_PATH = ROOT / "data" / "processed" / "SET_MEMBER1_FILENAME.csv"
TEXT_COLUMN = None  # example: the actual standardized text column name
LABEL_COLUMN = None  # example: the actual binary ground-truth column name
POSITIVE_LABEL = None  # exact observed label for the misinformation/risk class
SEED = 42

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Set DATA_PATH to Member 1's processed CSV: {DATA_PATH}")
if not all(value is not None for value in (TEXT_COLUMN, LABEL_COLUMN, POSITIVE_LABEL)):
    raise ValueError("Set TEXT_COLUMN, LABEL_COLUMN and POSITIVE_LABEL from Member 1's actual data")

data = pd.read_csv(DATA_PATH)
if LABEL_COLUMN not in data:
    raise KeyError(f"Missing label column {LABEL_COLUMN!r}; available: {list(data.columns)}")
print("Rows:", len(data), "| observed label counts:", data[LABEL_COLUMN].value_counts(dropna=False).to_dict())
if POSITIVE_LABEL not in set(data[LABEL_COLUMN].dropna()):
    raise ValueError("POSITIVE_LABEL must exactly match one observed label, including its type")
split = split_labeled_data(
    data, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN, random_state=SEED
)
print("Train / validation / test:", len(split.train), len(split.validation), len(split.test))


## Fit the unweighted classifier


In [ ]:
baseline = build_baseline(random_state=SEED)
baseline.fit(split.train[TEXT_COLUMN], split.train[LABEL_COLUMN])
print("Training label counts:", split.train[LABEL_COLUMN].value_counts().to_dict())
print("TF-IDF vocabulary size:", len(baseline.named_steps["tfidf"].vocabulary_))


## Validation, then held-out test

Accuracy alone can obscure performance on the minority class. Precision, recall and F1 use the explicitly supplied positive label. ROC AUC and average precision use positive-class probabilities.


In [ ]:
validation = evaluate_binary_classifier(
    baseline, split.validation[TEXT_COLUMN], split.validation[LABEL_COLUMN],
    positive_label=POSITIVE_LABEL, name="Unweighted validation",
)
test = evaluate_binary_classifier(
    baseline, split.test[TEXT_COLUMN], split.test[LABEL_COLUMN],
    positive_label=POSITIVE_LABEL, name="Unweighted test",
)
for result in (validation, test):
    print(result.name, result.metrics)
    print("[[TN, FP], [FN, TP]]:\n", result.confusion)
    display(plot_confusion(result))
    plt.close("all")
display(plot_roc_and_pr([test]))
plt.close("all")


## Reporting notes

Record the dataset version, date, deduplication rule, class mapping, split sizes and seed alongside real outputs in the manuscript. This is a text-only control; do not describe it as using Member 2's contextual features. Do not report a result until the actual processed file is present and the notebook has executed successfully.
